![image_1779960848021.png](./image_1779960848021.png "image_1779960848021.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date
from pyspark.sql import functions as f

# Initialize Spark session
spark = SparkSession.builder.appName("DeliveryDataFrame").getOrCreate()

# Input data
data = [
    (1, 101, "2023-01-01", "2023-01-01"),
    (2, 102, "2023-01-02", "2023-01-05"),
    (3, 103, "2023-01-03", "2023-01-03"),
    (4, 104, "2023-01-04", "2023-01-08"),
    (5, 105, "2023-01-05", "2023-01-05")
]

# Column names
columns = ["delivery_id", "customer_id", "order_date", "customer_pref_delivery_date"]

# Create DataFrame
df_delivery = spark.createDataFrame(data, columns)

# Convert string to DateType
df_delivery = df_delivery \
    .withColumn("order_date", to_date("order_date", "yyyy-MM-dd")) \
    .withColumn("customer_pref_delivery_date", to_date("customer_pref_delivery_date", "yyyy-MM-dd"))

# Show DataFrame
df_delivery.show()

In [0]:
# # result_df = (
# #     df_delivery.withColumn(
# #         "is_immediate",
# #         f.when(
# #             df_delivery.order_date == df_delivery.customer_pref_delivery_date, 1
# #         ).otherwise(0),
# #     )
# #     .withColumn("total_count",f.lit(f.count("is_immediate")*1.0/f.sum("is_immediate")))
# # )
# # display(result_df)

# total_count=df_delivery.count()
# immediate_count=df_delivery.filter(df_delivery.order_date==df_delivery.customer_pref_delivery_date).count()
# immediate_delivery_ratio = round((immediate_count * 1.0 / total_count)*100, 2)
# result_df = spark.createDataFrame([(immediate_delivery_ratio)], ["immediate_delivery_ratio"])
# display(result_df)


In [0]:
result_df = df_delivery.agg(
    f.round(
        (
            f.sum(
                f.when(
                    df_delivery.order_date == df_delivery.customer_pref_delivery_date, 1
                ).otherwise(0)
            )
            / f.count("*")
            * 100
        ),
        2,
    ).alias("immediate_delivery_ratio")
)
display(result_df)

In [0]:
# result_df = df_delivery.agg(
#     f.round(
#         f.sum(f.when(f.col("order_date") == f.col("customer_pref_delivery_date"), 1).otherwise(0)) / f.count("*"),
#         2
#     ).alias("immediate_delivery_ratio")
# )
# display(result_df)